In [8]:
import bs4
import requests

In [286]:
def getContent(url):
    # anti-scraping
    user_agent = "Mozilla/5.0 (Windows NT 10.0; WOW64; rv:68.0) Gecko/20100101 Firefox/68.0"
    
    response = requests.get(url, headers={'User-Agent': user_agent})
    if response.status_code == 200:
        # print(response.content.decode('utf-8'))
        return response.content
    else:
        print(f"Fail to get the url [{response.status_code}]")

In [513]:
def readPokemon(content):
    soup = bs4.BeautifulSoup(content, 'html.parser')
    card = soup.section
    # all type cards
    name = card.h1.get_text()

    # pokemon
    ## left box
    img = card.find('img', class_='fit')['src']

    reg = card.find('img', class_='img-regulation')['alt'] # regulation
    regImg = card.find('img', class_='img-regulation')['src']

    setNum = card.find('div', class_='subtext').get_text().strip()
    setCount = int(setNum.strip()[-3:])
    setNum = int(setNum.strip()[:3])

    if card.find('img', width='24'):
        rarityImg = card.find('img', width='24')['src']
        rarity = rarityImg.split('.')[0].split('rare_')[1]
    else:
        rarity = rarityImg = ''

    author = card.find('div', class_='author').get_text().strip().split('\n')[1]

    ### global pokedex
    pokedex = card.find('div', class_='card')
    if pokedex: # has pokedex
        [dexNum, dexClass] = pokedex.h4.get_text().strip().split('\u3000')
        dexNum = int(dexNum.split('.')[1])
        htAndWt = pokedex.p.get_text().split('：')
        height = float(htAndWt[1].split(' ')[0])
        weight = float(htAndWt[2].split(' ')[0])
        desc = pokedex.find_all('p')[1].get_text()
    else:
        [dexNum, dexClass, height, weight, desc] = [''] * 5

    ## right box
    stage = card.find('span', class_='type').get_text()
    if '\xa0' in stage:
        stage = stage.replace('\xa0', ' ')
    hp = card.find('span', class_='hp-num').get_text()
    hp = int(hp)
    pType = card.find('div', class_='td-r').find_all('span')[-1]['class'][0].split('-')[1]
    
    ### waza part
    part = content.split('<span class="hp-type">タイプ</span>')[1].split('</table>')[0].strip()
    soup = bs4.BeautifulSoup(part)
    wazaPart = bs4.BeautifulSoup(soup.prettify(formatter="minimal"))
    
    h2 = wazaPart.find_all('h2')
    skills = wazaPart.find_all('h4')
    p = wazaPart.find_all('p')
    # init
    [ability, abilityDesc, spRule] = [''] * 3 
    for area in h2:
        areaType = area.get_text().strip()
        if areaType == "特性":
            # ability
            # print('learning an ability')
            ability = skills[0].get_text().strip()
            abilityDesc = p[0].get_text().strip()
            del skills[0]

        elif areaType == "特別なルール":
            # special rule
            # print('learning a special rule')
            spRule = p[-1].get_text().strip()
            del p[-1]
    # waza
    # init
    [waza1Cost, waza1Name, waza1Damage, waza1Desc] = [''] * 4
    [waza2Cost, waza2Name, waza2Damage, waza2Desc] = [''] * 4

    waza1Cost = [span['class'][0].split('-')[1] for span in skills[0].find_all('span', class_='icon')]
    waza1 = skills[0].get_text().strip().split(' ')
    waza1Name = waza1[0].strip()
    waza1Damage = waza1[-1]
    waza1Desc = skills[0].find_next_sibling('p').get_text().strip()

    if len(skills) > 1:
        waza2Cost = [span['class'][0].split('-')[1] for span in skills[1].find_all('span', class_='icon')]
        waza2 = skills[1].get_text().strip().split(' ')
        waza2Name = waza2[0].strip()
        waza2Damage = waza2[-1]
        waza2Desc = skills[1].find_next_sibling('p').get_text().strip()
    
    ### table
    [weakType, weakValue, resistType, resistValue] = [''] * 4
    td = wazaPart.find_all('td')
    if td[0].find('span'):
        weakType = td[0].find('span')['class'][0].split('-')[1]
        weakValue = td[0].get_text().strip()

    if td[1].find('span'):
        resistType = td[1].find('span')['class'][0].split('-')[1]
        resistValue = td[1].get_text().strip()

    escape = len(td[2].find_all('span'))
    

    return [name, img, regulation, setNum, setCount, rarity, dexNum,
            dexClass, height, weight, desc, author, stage, hp, pType,
            ability, abilityDesc, waza1Cost, waza1Name, waza1Damage,
            waza1Desc, waza2Cost, waza2Name, waza2Damage, waza2Desc,
            weakType, weakValue, resistType, resistValue,
            escape, spRule]

In [517]:
# read pokemon card
# url = "https://www.pokemon-card.com/card-search/details.php/card/38407/regu/XY" # biidoru c
# url = "https://www.pokemon-card.com/card-search/details.php/card/38439/regu/XY" # zacian as
# url = "https://www.pokemon-card.com/card-search/details.php/card/38438/regu/XY" # mahoippu Vmax
# url = "https://www.pokemon-card.com/card-search/details.php/card/38426/regu/XY" # zekuromu r
# url = "https://www.pokemon-card.com/card-search/details.php/card/38419/regu/XY" # zaruudo V
# url = "https://www.pokemon-card.com/card-search/details.php/card/38423/regu/XY" # raiboruto c ability
# url = "https://www.pokemon-card.com/card-search/details.php/card/38310/regu/XY" # mew V resistance
# url = "https://www.pokemon-card.com/card-search/details.php/card/38300/regu/XY" # pikachu V
url = "https://www.pokemon-card.com/card-search/details.php/card/38304/regu/XY" # denryu

content = getContent(url)
cardId = 33001 # start: 33001, end: 38493
url = f'https://www.pokemon-card.com/card-search/details.php/card/{cardId}/regu/XY'
readPokemon(content)

['デンリュウ',
 '/assets/images/card_images/large/SD/038304_P_DENRYUU.jpg',
 'S3a',
 32,
 127,
 '',
 181,
 'ライトポケモン',
 1.4,
 61.5,
 'シッポは 強く 明るく 輝く。 船乗りたちの 道しるべ として 昔から 大切に されてきた。',
 'kodama',
 '2 進化',
 150,
 'electric',
 '',
 '',
 ['electric'],
 'せんこうだん',
 '50',
 '相手のバトルポケモンをこんらんにする。',
 ['electric', 'electric'],
 'ライトニングボール',
 '130',
 '',
 'fighting',
 '×2',
 '',
 '',
 2,
 '']

In [518]:
part = content.split('<span class="hp-type">タイプ</span>')[1].split('</table>')[0].strip()
soup = bs4.BeautifulSoup(part)
wazaPart = bs4.BeautifulSoup(soup.prettify(formatter="minimal"))

In [494]:

wazaPart

<html>
<body>
<span class="icon-grass icon">
</span>
<h2 class="mt20">
   ワザ
  </h2>
<h4>
<span class="icon-none icon">
</span>
<span class="icon-none icon">
</span>
   しばりつける
   <span class="f_right Text-fjalla">
    50
   </span>
</h4>
<p>
   次の相手の番、このワザを受けたポケモンは、にげられない。
  </p>
<h4>
<span class="icon-grass icon">
</span>
<span class="icon-grass icon">
</span>
   ジャングルライズ
   <span class="f_right Text-fjalla">
    100
   </span>
</h4>
<p>
   のぞむなら、自分の手札から基本エネルギーを2枚まで選び、ベンチポケモンに好きなようにつける。その後、つけたポケモンのHPをすべて回復する。
  </p>
<h2 class="mt20">
   特別なルール
  </h2>
<p>
   ポケモンVがきぜつしたとき、相手はサイドを2枚とる。
  </p>
<table cellpadding="0" cellspacing="0">
<tr>
<th>
     弱点
    </th>
<th>
     抵抗力
    </th>
<th>
     にげる
    </th>
</tr>
<tr>
<td>
<span class="icon-fire icon">
</span>
     ×2
    </td>
<td>
     --
    </td>
<td class="escape">
<span class="icon-none icon">
</span>
</td>
</tr>
</table>
</body>
</html>

In [495]:
h2 = wazaPart.find_all('h2')
h2

[<h2 class="mt20">
    ワザ
   </h2>,
 <h2 class="mt20">
    特別なルール
   </h2>]

In [496]:
h2 = wazaPart.find_all('h2')
skills = wazaPart.find_all('h4')
skills

[<h4>
 <span class="icon-none icon">
 </span>
 <span class="icon-none icon">
 </span>
    しばりつける
    <span class="f_right Text-fjalla">
     50
    </span>
 </h4>,
 <h4>
 <span class="icon-grass icon">
 </span>
 <span class="icon-grass icon">
 </span>
    ジャングルライズ
    <span class="f_right Text-fjalla">
     100
    </span>
 </h4>]

In [497]:
h2 = wazaPart.find_all('h2')
skills = wazaPart.find_all('h4')
p = wazaPart.find_all('p')
p

[<p>
    次の相手の番、このワザを受けたポケモンは、にげられない。
   </p>,
 <p>
    のぞむなら、自分の手札から基本エネルギーを2枚まで選び、ベンチポケモンに好きなようにつける。その後、つけたポケモンのHPをすべて回復する。
   </p>,
 <p>
    ポケモンVがきぜつしたとき、相手はサイドを2枚とる。
   </p>]

In [498]:
wazaCount = len(p) - len(h2) +1

h2 = wazaPart.find_all('h2')
skills = wazaPart.find_all('h4')
p = wazaPart.find_all('p')
# init
[ability, abilityDesc, spRule] = [''] * 3 
for area in h2:
    areaType = area.get_text().strip()
    if areaType == "特性":
        # ability
        print('learning an ability')
        ability = skills[0].get_text().strip()
        abilityDesc = p[0].get_text().strip()
        del skills[0]
        
    elif areaType == "特別なルール":
        # special rule
        print('learning a special rule')
        spRule = p[-1].get_text().strip()
        del p[-1]
        
[wazaCount, skills, p]

learning a special rule


[2,
 [<h4>
  <span class="icon-none icon">
  </span>
  <span class="icon-none icon">
  </span>
     しばりつける
     <span class="f_right Text-fjalla">
      50
     </span>
  </h4>,
  <h4>
  <span class="icon-grass icon">
  </span>
  <span class="icon-grass icon">
  </span>
     ジャングルライズ
     <span class="f_right Text-fjalla">
      100
     </span>
  </h4>],
 [<p>
     次の相手の番、このワザを受けたポケモンは、にげられない。
    </p>,
  <p>
     のぞむなら、自分の手札から基本エネルギーを2枚まで選び、ベンチポケモンに好きなようにつける。その後、つけたポケモンのHPをすべて回復する。
    </p>]]

In [499]:

wazaCost

['colorless', 'colorless']

In [507]:
# init
[waza1Cost, waza1Name, waza1Damage, waza1Desc] = [''] * 4
[waza2Cost, waza2Name, waza2Damage, waza2Desc] = [''] * 4

waza1Cost = [span['class'][0].split('-')[1] for span in skills[0].find_all('span', class_='icon')]
waza1 = skills[0].get_text().strip().split(' ')
waza1Name = waza1[0].strip()
waza1Damage = int(waza1[-1])
waza1Desc = skills[0].find_next_sibling('p').get_text().strip()

if len(skills) > 1:
    waza2Cost = [span['class'][0].split('-')[1] for span in skills[1].find_all('span', class_='icon')]
    waza2 = skills[1].get_text().strip().split(' ')
    waza2Name = waza2[0].strip()
    waza2Damage = int(waza2[-1])
    waza2Desc = skills[1].find_next_sibling('p').get_text().strip()
[waza1Cost, waza1Name, waza1Damage, waza1Desc, waza2Cost, waza2Name, waza2Damage, waza2Desc]

[['none', 'none'],
 'しばりつける',
 50,
 '次の相手の番、このワザを受けたポケモンは、にげられない。',
 ['grass', 'grass'],
 'ジャングルライズ',
 100,
 'のぞむなら、自分の手札から基本エネルギーを2枚まで選び、ベンチポケモンに好きなようにつける。その後、つけたポケモンのHPをすべて回復する。']

In [468]:
spRule

''

In [469]:
td = wazaPart.find_all('td')
td

[<td>
 <span class="icon-fire icon">
 </span>
      ×2
     </td>,
 <td>
      --
     </td>,
 <td class="escape">
 <span class="icon-none icon">
 </span>
 </td>]

In [472]:
[weakType, weakValue, resistType, resistValue] = [''] * 4

if td[0].find('span'):
    weakType = td[0].find('span')['class'][0].split('-')[1]
    weakValue = td[0].get_text().strip()

if td[1].find('span'):
    resistType = td[1].find('span')['class'][0].split('-')[1]
    resistValue = td[1].get_text().strip()
    
escape = len(td[2].find_all('span'))

[weakType, weakValue, resistType, resistValue, escape]

['fire', '×2', '', '', 1]

In [471]:
td[0].find('span')['class'][0].split('-')[1]

'fire'

In [56]:
# special engergy
twinEngergy = "https://www.pokemon-card.com/card-search/details.php/card/38482/regu/all"
card = readCard(twinEngergy)
card

<section class="Section">
<h1 class="Heading1 mt20">ツインエネルギー</h1>
<div class="Box">
<div class="LeftBox">
<img alt="ツインエネルギー" class="fit" src="/assets/images/card_images/large/S3a/038482_E_TSUINENERUGI.jpg"/>
<div class="subtext Text-fjalla">
<img alt="S3a" class="img-regulation" src="/assets/images/card/regulation_logo_1/S3a.gif"/> 076 / 076 <img src="/assets/images/card/rarity/ic_rare_u_c.gif" width="24"/> </div>
<span> </span>
<div class="author">
</div>
</div>
<div class="RightBox">
<div class="RightBox-inner">
<h2 class="mt20">特殊エネルギー</h2>
<p>このカードは、ポケモンについているかぎり、【無】エネルギー2個ぶんとしてはたらく。<br/>
<br/>
「ポケモンV・GX」についているなら、【無】エネルギー1個ぶんとしてはたらく。</p>
</div>
</div>
<div class="clear"></div>
</div>
</section>

In [57]:
# supporter
onion = "https://www.pokemon-card.com/card-search/details.php/card/38477/regu/XY"
card = readCard(onion)
card

<section class="Section">
<h1 class="Heading1 mt20">オニオン</h1>
<div class="Box">
<div class="LeftBox">
<img alt="オニオン" class="fit" src="/assets/images/card_images/large/S3a/038477_T_ONION.jpg"/>
<div class="subtext Text-fjalla">
<img alt="S3a" class="img-regulation" src="/assets/images/card/regulation_logo_1/S3a.gif"/> 071 / 076 <img src="/assets/images/card/rarity/ic_rare_u_c.gif" width="24"/> </div>
<span> </span>
<div class="author">
<h4>イラストレーター</h4>
<a href="/card-search/index.php?regulation_illust=XY&amp;illust=take">take</a> </div>
</div>
<div class="RightBox">
<div class="RightBox-inner">
<h2 class="mt20">サポート</h2>
<p>自分の山札を3枚引く。その後、自分の手札を3枚まで選び、トラッシュする。（必ず1枚はトラッシュする。）</p>
<p>サポートは、自分の番に1枚しか使えない。</p>
</div>
</div>
<div class="clear"></div>
</div>
</section>